# Metadata and Schema Management

In [ ]:
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import *

MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
DATABASE = "default"
BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"
BUCKET_GOLD = "gold"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppM6Class02") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.eventLog.dir", "file:/home/jovyan/work/spark-logs") \
    .config("spark.history.fs.logDirectory", "file:/home/jovyan/work/spark-logs") \
    .config("log4j.rootCategory", "INFO, console") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true") \
    .config("spark.executor.cleanupOnShutdown", "true") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

## Let's create a new table in the gold layer, from an existing one.

In [ ]:
table_gold = "hotel_booking_gold_zordering"
location_gold = f"s3a://{BUCKET_GOLD}/delta/{table_gold}"

table_gold_schema = "hotel_booking_schema_evolution"
location_gold_schema = f"s3a://{BUCKET_GOLD}/delta/{table_gold_schema}"

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_gold}
    USING DELTA
    LOCATION '{location_gold}'
""")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_gold_schema}
    USING DELTA
    LOCATION '{location_gold_schema}'
    AS 
    SELECT 
        * 
    FROM 
        {DATABASE}.{table_gold}
""")

In [ ]:
df = spark.sql(f"SELECT * FROM {DATABASE}.{table_gold_schema}")

In [ ]:
df.printSchema()

In [ ]:
df.limit(10).show(truncate=False)

## Attempting to 'append' without mergeSchema -> should fail due to schema mismatch

In [ ]:
new_df = df.withColumn("meal_plan", F.lit("Breakfast"))

In [ ]:
new_df.printSchema()

In [ ]:
new_df.limit(10).show(truncate=False)

In [ ]:
%%time
(
    new_df.write.format("delta")
    .mode("append")
    .save(location_gold_schema)
)

## Use `mergeSchema` to enable schema evolution

In [ ]:
new_df.write.format("delta").option("mergeSchema", "true").mode("append").save(location_gold_schema)

## Read the data again to verify the final schema

In [ ]:
print(">>> Final scheme after evolution:\n")
spark.sql(f"SELECT * FROM {DATABASE}.{table_gold_schema}").printSchema()

In [ ]:
print(">>> Data after schema evolution:\n")
spark.sql(f"SELECT * FROM {DATABASE}.{table_gold_schema}").limit(10).show(truncate=False)

## Now let's change the DataType of a column and try to evolve its schema.

In [ ]:
df_converted = df.withColumn(
    "reservation_status_date",
    df["reservation_status_date"].cast("string")  # ou StringType()
)

In [ ]:
df_converted.write.format("delta").option("mergeSchema", "true").mode("append").save(location_gold_schema)

## Overwriting the data to change the DataType of the `reservation_status_date` column using the option: `overwriteSchema`

In [ ]:
(
    df_converted.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "True")
    .save(location_gold_schema)
)

In [ ]:
print(">>> Final schema after DataType evolution:\n")
spark.sql(f"SELECT * FROM {DATABASE}.{table_gold_schema}").printSchema()

In [ ]:
print(">>> Data after DataType evolution:\n")
spark.sql(f"SELECT * FROM {DATABASE}.{table_gold_schema}").limit(10).show(truncate=False)

In [ ]:
spark.stop()